# Exercise 05. Pandas optimizations

1. Read the `fines.csv` file that you saved in the previous exercise.

In [ ]:
import pandas as pd
df=pd.read_csv('../data/fines.csv')
df=df.rename(columns={'fines': 'Year'})
df

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,1989
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1984
3,X582HE161RUS,2.0,2000.0,Ford,Focus,2015
4,E34877152RUS,2.0,6100.0,Ford,Focus,2014
...,...,...,...,...,...,...
931,A001AA99RUS,2.0,1500.0,Tesla,Model S,2021
932,B002BB77RUS,1.0,500.0,Kia,Rio,2019
933,C003CC152RUS,2.0,2500.0,Haval,Jolion,2022
934,D004DD50RUS,1.0,1000.0,Geely,Coolray,2023


2. Iterations: in all the following subtasks, you need to calculate `fines/refund*year` for each row. Create a new column with the calculated data. Measure the time using the magic command `%%timeit` in the cell.
   - Write a function that loops through the dataframe using `for i in range(0, len(df))`, `iloc`, and `append()` to a list. Assign the result of the function to a new column in the dataframe.

In [31]:
%%timeit
loop_calc=[]
for i in range(len(df)-1):
    temp=df['Fines'].iloc[i]/(df['Refund'].iloc[i]*df['Year'].iloc[i])
    loop_calc.append(temp)
#print(f"result with using loop alg :{loop_calc}")
loop_column=pd.Series(loop_calc, name='Loop')
df_loop=pd.concat([df,loop_column], axis=1)
df_loop


22.6 ms ± 4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


 - Do it using `iterrows()`.

In [32]:
%%timeit
iterrows_calc=[]
for index, row in df.iterrows():
    temp_iter=row['Fines']/(row['Refund']*row['Year'])
    iterrows_calc.append(temp_iter)
df_iter=df.copy()
df_iter['Iter']=iterrows_calc
#df_iter


55.3 ms ± 9.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


 - Do it using `apply()` and a lambda function.

In [33]:
%%timeit
df_apply=df.copy()
df_apply['Apply']=df.apply(lambda row: row['Fines'] / (row['Refund']*row['Year']), axis=1)
df_apply


8.88 ms ± 224 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


 - Do it using `Series` objects from the dataframe.

In [49]:
#%%timeit
df_series=df.copy()
df_series['Series']=df_series['Fines']/df_series['Refund']*df_series['Year']
df_series

,Refund,Fines,Make,Model,Year,Series
CarNumber,,,,,,
Y163O8161RUS,2.0,3200.0,Ford,Focus,1989,3182400.0
E432XX77RUS,1.0,6500.0,Toyota,Camry,1995,12967500.0
7184TT36RUS,1.0,2100.0,Ford,Focus,1984,4166400.0
X582HE161RUS,2.0,2000.0,Ford,Focus,2015,2015000.0
E34877152RUS,2.0,6100.0,Ford,Focus,2014,6142700.0
...,...,...,...,...,...,...
A001AA99RUS,2.0,1500.0,Tesla,Model S,2021,1515750.0
B002BB77RUS,1.0,500.0,Kia,Rio,2019,1009500.0
C003CC152RUS,2.0,2500.0,Haval,Jolion,2022,2527500.0


- Do it as in the previous subtask, but use the method `.values`.

In [37]:
%%timeit
df_val=df.copy()
df_val['Values']=df_val['Fines'].values/df_val['Refund'].values*df_val['Year'].values
#df_val

337 μs ± 37.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 3. Indexing: measure the time using the magic command `%%timeit` in the cell.
   - Get a row for a specific `CarNumber`, for example, "O136HO197RUS."
   - Set the index in your dataframe with `CarNumber`.
   - Again, get a row for the same `CarNumber`.

In [43]:
%%timeit
my_number='O136HO197RUS'
for i in range(len(df)-1):
    if df['CarNumber'].loc[i]==my_number:
        #print("Find it")
        break


8.13 ms ± 757 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [52]:
%%timeit
my_number='O136HO197RUS'
#df.set_index('CarNumber', inplace=True)
res=df.loc[my_number]


66.6 μs ± 3.04 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 4. Downcasting:
   - Run `df.info(memory_usage='deep')`, and pay attention to the Dtype and memory usage.
   - Make a `copy()` of your initial dataframe into another dataframe, `optimized_df`.
   - Downcast from `float64` to `float32` for all columns.
   - Downcast from `int64` to the smallest numerical Dtype possible.
   - Run `info(memory_usage='deep')` for your new dataframe. Pay attention to the Dtype and memory usage.

In [53]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 936 entries, Y163O8161RUS to E005EE178RUS
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Refund  936 non-null    float64
 1   Fines   936 non-null    float64
 2   Make    936 non-null    object 
 3   Model   924 non-null    object 
 4   Year    936 non-null    int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 208.1 KB


In [55]:
oprimized_df=df.copy()
oprimized_df[oprimized_df.select_dtypes(include='float64').columns]=oprimized_df.select_dtypes(include=['float64']).astype('float32')
for col in oprimized_df.select_dtypes(include='int64').columns:
    oprimized_df[col] = pd.to_numeric(oprimized_df[col], downcast='integer')


In [57]:
oprimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 936 entries, Y163O8161RUS to E005EE178RUS
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Refund  936 non-null    float32
 1   Fines   936 non-null    float32
 2   Make    936 non-null    object 
 3   Model   924 non-null    object 
 4   Year    936 non-null    int16  
dtypes: float32(2), int16(1), object(2)
memory usage: 195.3 KB


## 5. Categories:
   - Change the `object` type columns to `category`.
   - This time, check the memory usage. It will probably decrease by 2–3 times compared to the initial dataframe.

In [58]:
oprimized_df[oprimized_df.select_dtypes(include='object').columns]=oprimized_df.select_dtypes(include=['object']).astype('category')

In [59]:
oprimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 936 entries, Y163O8161RUS to E005EE178RUS
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   Refund  936 non-null    float32 
 1   Fines   936 non-null    float32 
 2   Make    936 non-null    category
 3   Model   924 non-null    category
 4   Year    936 non-null    int16   
dtypes: category(2), float32(2), int16(1)
memory usage: 101.1 KB


## 6. Memory clean:
   - Using the library `gc` and the command `%reset_selective`, clean the memory of your initial dataframe only.

In [60]:
import gc 
%reset_selective -f "^df$"

gc.collect()

4566